### 파이썬의 핵심 : 모든 것은 객체(주소)다
- 파이썬에서 변수는 데이터를 직접 저장하지 않는다.
- 데이터가 담긴 객체의 '메모리 주소'를 가리키는 이정표(참조) 역할을 할 뿐이다.

- 예를 들어, a = [1, 2]라고 하면 a라는 변수에 리스트가 저장되는 것이 아니라, 
- 메모리 어딘가에 생성된 [1, 2]라는 객체를 a가 가리키게 됩니다.

- 여기서 "가변 객체(Mutable Object)"가 등장하면서 복사 문제가 복잡해집니다.
- 리스트(list), 딕셔너리(dict), 집합(set) 등은 생성된 후에도 내부 값을 바꿀 수 있는 가변 객체입니다.
- 가변 객체 내부에 또 다른 가변 객체가 들어가는 '중첩 구조'가 될 때, 복사의 종류에 따라 메모리 배치 방식이 달라집니다.

### 얕은 복사(Shallow Copy)
- "껍데기(가장 바깥 객체)만 새로 만들고, 알맹이(내부 객체)는 주소만 복사한다."
- 객체와의 관계: 새로운 객체를 하나 만들긴 하지만, 그 안에 담기는 요소들은 원본 객체 안에 있던 요소들의 메모리 주소를 그대로 복사해 옵니다.
- 문제점: 만약 리스트 안에 리스트가 있는 구조라면, 내부 리스트의 주소가 같기 때문에 복사본에서 내부 리스트를 수정하면 원본까지 동시에 수정되어 버립니다.

### 깊은 복사(Deep Copy)
- "껍데기뿐만 아니라 알맹이까지, 참조된 모든 객체를 재귀적으로 새로 만든다."
- 객체와의 관계: 원본 객체가 가리키고 있는 모든 중첩 객체들을 추적하여, 
- 메모리의 완전히 새로운 공간에 독립된 객체들로 똑같이 복제합니다.
- 결과: 원본 객체와 복사본 객체 사이에 그 어떤 메모리 주소 공유도 일어나지 않으므로, 
- 서로 완벽하게 독립됩니다.

- "그럼 안전하게 항상 깊은 복사(deepcopy)만 쓰면 되지 않나요?"라고 생각할 수 있습니다. 
- 하지만 깊은 복사는 모든 객체를 새로 만들기 때문에 메모리를 많이 차지하고 속도가 느립니다. 
- 구조가 복잡하지 않거나 내부 요소가 변하지 않는 데이터라면 얕은 복사를 쓰는 것이 훨씬 효율적입니다.
- 데이터의 객체 구조를 파악하고, 어떤 복사를 적용할지 올바르게 선택하는 안목이 중요합니다.

In [2]:
import copy

class Passenger:
    """승객을 나타내는 클래스"""
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return self.name


class Bus:
    """버스를 나타내는 클래스"""
    def __init__(self, bus_num, passengers):
        self.bus_num = bus_num
        self.passengers = passengers  # 승객 객체들이 담길 리스트

    def pick(self, passenger_name):
        """승객이 버스에 타는 메서드 (pick)"""
        new_passenger = Passenger(passenger_name)
        self.passengers.append(new_passenger)
        print(f"[{self.bus_num}번 버스] 👤 {passenger_name} 탑승 -> 현재 승객: {self.passengers}")

    def drop(self, passenger_name):
        """승객이 버스에서 내리는 메서드 (drop)"""
        # 이름이 일치하는 승객 찾기 : drop 명령에 해당하는 승객이 없는 경우 대비 (260605)
        target = None
        for p in self.passengers:
            if p.name == passenger_name:
                target = p
                break
        
        if target:
            self.passengers.remove(target)
            print(f"[{self.bus_num}번 버스] 🚪 {passenger_name} 하차 -> 현재 승객: {self.passengers}")
        else:
            print(f"[{self.bus_num}번 버스] {passenger_name} 이름의 승객이 없습니다.")

In [3]:
# 복사본 생성 및 초기화

# 1. 원본 100번 버스 생성 (초기 승객: 철수, 영희)
bus_100 = Bus("100", [Passenger("철수"), Passenger("영희")])

# 2. 얕은 복사와 깊은 복사 진행
shallow_bus = copy.copy(bus_100)
deep_bus = copy.deepcopy(bus_100)

# 복사된 버스의 번호 변경 (구분용)
shallow_bus.bus_num = "200 (얕은 복사)"
deep_bus.bus_num = "300 (깊은 복사)"

In [6]:
print(bus_100.bus_num, bus_100.passengers)
print(shallow_bus.bus_num, shallow_bus.passengers)
print(deep_bus.bus_num, deep_bus.passengers)

100 [철수, 영희]
200 (얕은 복사) [철수, 영희]
300 (깊은 복사) [철수, 영희]


In [7]:
print("=== 얕은 복사 버스(200번) 조작 ===")
# 200번 버스에 '민수'가 타고, '철수'가 내립니다.
shallow_bus.pick("민수")
shallow_bus.drop("철수")


print("\n[결과 확인]")
print(f"100번 버스(원본) 승객: {bus_100.passengers}")
print(f"200번 버스(얕은) 승객: {shallow_bus.passengers}")

=== 얕은 복사 버스(200번) 조작 ===
[200 (얕은 복사)번 버스] 👤 민수 탑승 -> 현재 승객: [철수, 영희, 민수]
[200 (얕은 복사)번 버스] 🚪 철수 하차 -> 현재 승객: [영희, 민수]

[결과 확인]
100번 버스(원본) 승객: [영희, 민수]
200번 버스(얕은) 승객: [영희, 민수]


In [ ]:
# copy.copy는 Bus 객체 자체는 100번과 200번으로 분리해 주었지만, 
# 버스 안의 passengers 리스트(메모리 주소)는 그대로 공유하게 만들었습니다. 
# 따라서 200번 버스에서 사람이 타고 내리면 100번 버스 승객도 똑같이 바뀝니다.

In [8]:
# 깊은 복사 버스 조작 (300번 버스)

print("\n=== 깊은 복사 버스(300번) 조작 ===")
# 300번 버스에 '길동'이가 타고, '영희'가 내립니다.
deep_bus.pick("길동")
deep_bus.drop("영희")

print("\n[결과 확인]")
print(f"100번 버스(원본) 승객: {bus_100.passengers}")
print(f"300번 버스(깊은) 승객: {deep_bus.passengers}")



=== 깊은 복사 버스(300번) 조작 ===
[300 (깊은 복사)번 버스] 👤 길동 탑승 -> 현재 승객: [철수, 영희, 길동]
[300 (깊은 복사)번 버스] 🚪 영희 하차 -> 현재 승객: [철수, 길동]

[결과 확인]
100번 버스(원본) 승객: [영희, 민수]
300번 버스(깊은) 승객: [철수, 길동]


In [ ]:
# copy.deepcopy는 Bus 객체뿐만 아니라, 
# 내부에 있던 passengers 리스트와 리스트 안의 Passenger 객체들까지 전부 새로 복사해서 
# 완전히 독립된 버스를 만듭니다. 
# 그래서 300번 버스에서 어떤 일이 일어나든 원본 100번 버스에는 아무런 영향이 없습니다.